# Segmentation benchmarking

This notebook compares predicted instance-segmentation masks with corresponding ground-truth (GT) masks. One-to-one object matching is performed using intersection over union (IoU), and segmentation performance is evaluated across IoU thresholds from 0.5 to 0.9. Reported metrics include precision, recall, F1 score, mean IoU over matched objects, and the 95th-percentile Hausdorff distance (HD95) measured in pixels.
As input, both the GT and the predicted masks should be loaded as .tif files.


In [ ]:
import os

# Set the working directory
os.chdir('your_path')

In [ ]:
import tifffile as tiff

gt_mask = tiff.imread('your_gt_mask.tif')
pred_mask = tiff.imread('your_predicted_mask.tif')


In [ ]:
import numpy as np
from skimage.measure import label, regionprops
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# gt_mask, pred_mask: 2D numpy arrays of ints or bools
#  where each connected object has nonzero pixels
#  and background is zero.

gt_lbl = gt_mask.astype(np.int32)  # keep original labels
pr_lbl = pred_mask.astype(np.int32)   # integer labels 1..N_pr
props_gt = regionprops(gt_lbl)
props_pr = regionprops(pr_lbl)
N_gt = len(props_gt)
N_pr = len(props_pr)


In [ ]:
# Precompute pixel‐sets for faster intersection/union
gt_sets = {}
for i, p in enumerate(props_gt):
    bbox = p.bbox  # (min_row, min_col, max_row, max_col)
    gt_crop = gt_lbl[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    obj_pixels = np.argwhere(gt_crop == p.label)
    obj_pixels += np.array([bbox[0], bbox[1]])  # restore absolute coords
    gt_sets[i] = set(map(tuple, obj_pixels))

pr_sets = {}
for j, p in enumerate(props_pr):
    bbox = p.bbox
    pr_crop = pr_lbl[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    obj_pixels = np.argwhere(pr_crop == p.label)
    obj_pixels += np.array([bbox[0], bbox[1]])
    pr_sets[j] = set(map(tuple, obj_pixels))

iou_mat = np.zeros((len(gt_sets), len(pr_sets)), dtype=float)
for i in gt_sets:
    for j in pr_sets:
        inter = len(gt_sets[i] & pr_sets[j])
        union = len(gt_sets[i] | pr_sets[j])
        iou_mat[i, j] = inter / union if union > 0 else 0.0


In [ ]:
# Hungarian method on a cost matrix = -IoU
cost = -iou_mat
row_idx, col_idx = linear_sum_assignment(cost)

In [ ]:
from skimage.segmentation import find_boundaries

def hd95(mask_gt, mask_pr):
    # get boundary pixel coords
    b_gt = np.argwhere(find_boundaries(mask_gt, mode="inner"))
    b_pr = np.argwhere(find_boundaries(mask_pr, mode="inner"))
    if len(b_gt) == 0 or len(b_pr) == 0:
        return np.nan
    # pairwise distances
    D = cdist(b_gt, b_pr)
    # directed distances
    d1 = np.min(D, axis=1)  # each GT boundary → pred
    d2 = np.min(D, axis=0)  # each pred boundary → GT
    # 95th percentile symmetric
    return max(np.percentile(d1, 95), np.percentile(d2, 95))


# Evaluate segmentation performance across IoU thresholds 0.5–0.9
thresholds = np.arange(0.5, 1.0, 0.1)

print(f"Ground-truth objects: {N_gt}")
print(f"Predicted objects:    {N_pr}\n")
print("IoU thr |  TP |  FP |  FN | Precision | Recall |   F1   | Mean IoU | Mean HD95")
print("--------|-----|-----|-----|-----------|--------|--------|----------|----------")

for thr in thresholds:
    matches = [(r, c) for r, c in zip(row_idx, col_idx) if iou_mat[r, c] >= thr]

    TP = len(matches)
    FN = N_gt - TP
    FP = N_pr - TP

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0)

    ious = [iou_mat[r, c] for r, c in matches]
    mean_iou = np.mean(ious) if ious else np.nan

    hd95_list = []
    for r, c in matches:
        gt_obj = (gt_lbl == props_gt[r].label)
        pr_obj = (pr_lbl == props_pr[c].label)
        hd95_list.append(hd95(gt_obj, pr_obj))
    mean_hd95 = np.nanmean(hd95_list) if hd95_list else np.nan

    print(
        f" {thr:0.1f}    | {TP:3d} | {FP:3d} | {FN:3d} |"
        f"   {precision:0.4f}  | {recall:0.4f} | {f1:0.4f} |"
        f"  {mean_iou:0.4f}  |  {mean_hd95:0.4f}"
    )